In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score

from scipy.stats import pearsonr

import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# embeddings generados previamente
X = np.load("protein_embeddings.npy")

# targets
targets = pd.read_csv("targets.csv")

y = targets[["pTM", "ipTM"]].values

print("Embeddings shape:", X.shape)
print("Targets shape:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
pca = PCA(n_components=256)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("PCA shape:", X_train_pca.shape)

In [ ]:
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of components")
plt.ylabel("Explained variance")
plt.show()

In [ ]:
model_ptm = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

model_ptm.fit(X_train_pca, y_train[:,0])

In [ ]:
model_iptm = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

model_iptm.fit(X_train_pca, y_train[:,1])

In [ ]:
pred_ptm = model_ptm.predict(X_test_pca)
pred_iptm = model_iptm.predict(X_test_pca)

preds = np.vstack([pred_ptm, pred_iptm]).T

In [ ]:
mse_ptm = mean_squared_error(y_test[:,0], pred_ptm)
mse_iptm = mean_squared_error(y_test[:,1], pred_iptm)

print("MSE pTM:", mse_ptm)
print("MSE ipTM:", mse_iptm)

In [ ]:
print("R2 pTM:", r2_score(y_test[:,0], pred_ptm))
print("R2 ipTM:", r2_score(y_test[:,1], pred_iptm))

In [ ]:
print("Pearson pTM:", pearsonr(y_test[:,0], pred_ptm))
print("Pearson ipTM:", pearsonr(y_test[:,1], pred_iptm))

In [ ]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test[:,0], y=pred_ptm)

plt.xlabel("True pTM")
plt.ylabel("Predicted pTM")

plt.plot([0,1],[0,1], color="red")

plt.show()

In [ ]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test[:,1], y=pred_iptm)

plt.xlabel("True ipTM")
plt.ylabel("Predicted ipTM")

plt.plot([0,1],[0,1], color="red")

plt.show()